# 🔍 KIỂM TRA DỮ LIỆU 90 LOÀI CÂY (CHẠY RIÊNG KHÔNG ẢNH HƯỞNG TIẾN ĐỘ TRAIN)
**Notebook này chạy bằng CPU thông thường (không cần GPU), giúp bạn xem ảnh thực tế của từng loài cây để kiểm tra xem có bị nhầm lẫn hay lưu lộn tệp không.**

> ⚠️ **Lưu ý:** Bạn cứ để file train chạy bình thường trên GPU. File kiểm tra này chạy hoàn toàn độc lập!

In [ ]:
# === BƯỚC 1: KẾT NỐI DRIVE & MỞ BỘ DỮ LIỆU ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import json

PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
ZIP_PATH = PROJECT / 'merged_v1/merged_90_split.zip'
LABELS_PATH = PROJECT / 'labels/labels_vi.json'

assert ZIP_PATH.is_file(), f'Không tìm thấy file: {ZIP_PATH}'

# Đọc tên tiếng Việt của 90 loài
vi_map = {}
if LABELS_PATH.exists():
    with open(LABELS_PATH, encoding='utf-8') as f:
        for item in json.load(f):
            vi_map[item['original_label']] = item['vietnamese_name']

print('✅ Đã kết nối Google Drive thành công!')


In [ ]:
# === BƯỚC 2: CHỌN VÀ XEM ẢNH CỦA BẤT KỲ LOÀI NÀO (KHÔNG CẦN GIẢI NÉN TOÀN BỘ 6GB) ===
# Đọc trực tiếp từ file zip nên chạy cực nhanh (chỉ mất 1-2 giây)!
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image
import io
import random

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    all_files = archive.namelist()

# Lấy danh sách 90 loài
class_set = set()
for f in all_files:
    parts = f.split('/')
    if len(parts) >= 3 and parts[0] in ['train', 'val', 'test'] and parts[2].endswith('.jpg'):
        class_set.add(parts[1])

class_list = sorted(list(class_set))

# Tạo menu chọn loài cây
dropdown = widgets.Dropdown(
    options=[(f"{i:02d}. {vi_map.get(c, c)} ({c})", c) for i, c in enumerate(class_list)],
    description='Chọn cây:',
    layout={'width': '550px'}
)

output_area = widgets.Output()

def show_plant(change):
    cls = change['new']
    with output_area:
        output_area.clear_output()
        with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
            matching_files = [f for f in all_files if f'/{cls}/' in f and f.endswith('.jpg')]
            samples = random.sample(matching_files, min(8, len(matching_files)))
            
            fig, axes = plt.subplots(2, 4, figsize=(15, 7))
            for f_path, ax in zip(samples, axes.flatten()):
                data = archive.read(f_path)
                img = Image.open(io.BytesIO(data))
                ax.imshow(img.convert('RGB'))
                ax.set_title(f_path.split('/')[-1][:12], fontsize=8)
                ax.axis('off')
            
            vi_title = vi_map.get(cls, cls)
            plt.suptitle(f"{vi_title} — {cls} (Tổng: {len(matching_files)} ảnh)", fontsize=14, color='green', fontweight='bold')
            plt.tight_layout()
            plt.show()

dropdown.observe(show_plant, names='value')
display(dropdown)
display(output_area)

# Kích hoạt hiển thị loài đầu tiên
dropdown.value = class_list[0]


In [ ]:
# === BƯỚC 3: XUẤT FILE HTML DUYỆT NHANH CẢ 90 LOÀI CÂY (TẢI VỀ MỞ BẰNG CHROME) ===
import base64
from google.colab import files

print('Đang tổng hợp trang web duyệt ảnh 90 loài... Vui lòng đợi khoảng 15 giây...')

html = '''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Kiểm tra ảnh 90 loài cây</title>
<style>
body{font-family:Segoe UI,sans-serif;background:#f4f6f8;padding:20px;margin:0}
h1{text-align:center;color:#1b5e20;margin-bottom:5px}
.subtitle{text-align:center;color:#666;margin-bottom:25px}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(320px,1fr));gap:16px;max-width:1400px;margin:auto}
.card{background:#fff;border-radius:10px;padding:12px;box-shadow:0 2px 8px rgba(0,0,0,0.08);border:1px solid #e0e0e0}
.card h3{margin:0 0 4px 0;color:#2e7d32;font-size:16px}
.card p{margin:0 0 10px 0;color:#777;font-size:12px;font-style:italic}
.imgs{display:grid;grid-template-columns:repeat(4,1fr);gap:6px}
.imgs img{width:100%;height:75px;object-fit:cover;border-radius:6px;border:1px solid #eee}
</style></head><body>
<h1>🌿 DANH SÁCH KIỂM TRA 90 LOÀI CÂY KIỂNG LÁ</h1>
<div class="subtitle">Mỗi loài gồm 4 ảnh thực tế để bạn đối chiếu xem tên cây có khớp với ảnh trong kho không.</div>
<div class="grid">'''

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    for i, cls in enumerate(class_list):
        vi_name = vi_map.get(cls, cls)
        matching = [f for f in all_files if f'/{cls}/' in f and f.endswith('.jpg')]
        samples = random.sample(matching, min(4, len(matching)))
        
        html += f'<div class="card"><h3>{i:02d}. {vi_name}</h3><p>{cls} ({len(matching)} ảnh)</p><div class="imgs">'
        for s in samples:
            raw_data = archive.read(s)
            img = Image.open(io.BytesIO(raw_data))
            img.thumbnail((120, 120))
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=75)
            b64 = base64.b64encode(buf.getvalue()).decode()
            html += f'<img src="data:image/jpeg;base64,{b64}">'
        html += '</div></div>'

html += '</div></body></html>'

out_file = Path('/content/gallery_kiem_tra_90_loai.html')
out_file.write_text(html, encoding='utf-8')
print('✅ Đã tạo xong file! Đang tự động tải về máy...')
files.download(str(out_file))
